In [ ]:
# OpenCode / OpenAI-compatible notebook setup.
# Install once in the notebook's Python environment: pip install openai python-dotenv pyyaml
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import os as _nb_os

from pathlib import Path as _NotebookPath

from dotenv import load_dotenv as _notebook_load_dotenv
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Paths
#
CHAPTER = "s10_task_system"

PROJECT_ROOT = next(
    (
        directory
        for start in [_NotebookPath.cwd(), *_NotebookPath.cwd().parents]
        for directory in [start]
        if ((directory / CHAPTER / "code.py").is_file())
    ),
    None,
)

if (PROJECT_ROOT is None):

    raise (FileNotFoundError(
        f"Start Jupyter from the project root or {CHAPTER}/ "
        "so code.py can be located"
    ))
#

_notebook_load_dotenv(PROJECT_ROOT / ".env", override=True)
_nb_os.chdir(PROJECT_ROOT)

__file__ = str(PROJECT_ROOT / CHAPTER / "code.py")
#
###################################################################################################
###################################################################################################
################################################################################################### Interactive Entry Point
#
# Run All must not start a blocking CLI. Enable this explicitly if needed.
RUN_AGENT_INTERACTIVELY = False
#
###################################################################################################
###################################################################################################
################################################################################################### OpenCode Configuration
#
# The existing ANTHROPIC_* environment names remain supported.
_notebook_key   = (_nb_os.getenv("OPENCODE_API_KEY")  or  _nb_os.getenv("ANTHROPIC_API_KEY"))
_notebook_model = (_nb_os.getenv("OPENCODE_MODEL")  or  _nb_os.getenv("MODEL_ID"))
_notebook_url   = (
    _nb_os.getenv("OPENCODE_BASE_URL")
    or  _nb_os.getenv("ANTHROPIC_BASE_URL")
    or  "https://opencode.ai/inference/openai/v1"
).rstrip("/")

if (_notebook_url.endswith("/chat/completions")):

    _notebook_url = _notebook_url[:-len("/chat/completions")]
#

if (not _notebook_key):

    raise (RuntimeError(
        "Set OPENCODE_API_KEY (or ANTHROPIC_API_KEY) in the root .env"
    ))
#

if (not _notebook_model):

    raise (RuntimeError(
        "Set MODEL_ID (or OPENCODE_MODEL) in the root .env"
    ))
#

_nb_os.environ["OPENCODE_API_KEY"]   = _notebook_key
_nb_os.environ["OPENCODE_BASE_URL"]  = _notebook_url
_nb_os.environ["ANTHROPIC_BASE_URL"] = _notebook_url
_nb_os.environ["MODEL_ID"]          = _notebook_model

print(f"{CHAPTER}: OpenCode via OpenAI-compatible API, model={_notebook_model}")
#


In [ ]:
# Bridge the lesson's Anthropic-style message interface to OpenCode's OpenAI API.
# Legacy chapter loaders import "anthropic" dynamically; this mapping serves those
# imports inside the current notebook kernel without changing the .py files.
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import json  as _compat_json
import sys   as _compat_sys
import types as _compat_types

from openai import OpenAI
import uuid
#
###################################################################################################
###################################################################################################
################################################################################################### Content Conversion
#
def _compat_field(block:object, key:str, default=None):

    """
        Read a response field from either a plain dictionary or provider object.

        The chapter code consumes both forms; this helper keeps those access
        patterns equivalent without mutating the original response.
    """

    return ((block.get(key, default))
            if (isinstance(block, dict))
            else (getattr(block, key, default)))
#


def _compat_text(value:object) -> str:

    """
        Convert Anthropic-style content blocks to Chat Completions text.

        Exclude tool-use blocks: their identifiers and arguments are translated
        separately so that subsequent tool results can be matched correctly.
    """

    if (isinstance(value, str)):

        return (value)
    #
    if (value is None):

        return ("")
    #
    if (isinstance(value, list)):

        return ("\n".join(
            _compat_text(_compat_field(item, "text", _compat_field(item, "content", "")))
            for item in value
            if (_compat_field(item, "type") != "tool_use")
        ))
    #
    if (isinstance(value, dict)):

        return (_compat_text(value.get(
            "text", value.get("content", _compat_json.dumps(value))
        )))
    #
    return (str(value))
#


def _compat_history(messages:list, system:str|list|None=None) -> list:

    """
        Translate chapter conversation history for OpenAI Chat Completions.

        Preserve every assistant tool-call ID and pair it with the corresponding
        tool response. The system prompt is inserted before conversational turns.
    """

    result = []

    if (system):

        result.append({"role": "system", "content": _compat_text(system)})
    #

    for message in messages:

        role    = _compat_field(message, "role")
        content = _compat_field(message, "content", "")

        if (role == "assistant"):

            if (not isinstance(content, list)):

                result.append({"role": "assistant", "content": _compat_text(content)})
                continue
            #

            text_blocks = []
            tool_calls  = []

            for block in content:

                kind = _compat_field(block, "type")

                if (kind == "tool_use"):

                    arguments = _compat_field(block, "input", {})

                    tool_calls.append({
                        "id"       : _compat_field(block, "id"),
                        "type"     : "function",
                        "function" : {
                            "name"      : _compat_field(block, "name"),
                            "arguments" : _compat_json.dumps(arguments, ensure_ascii=False),
                        },
                    })
                #
                elif (kind == "text"):

                    text_blocks.append(_compat_field(block, "text", ""))
                #
            #

            converted = {"role": "assistant", "content": "\n".join(text_blocks) or None}

            if (tool_calls):

                converted["tool_calls"] = tool_calls
            #
            if (converted["content"] is None  and  not tool_calls):

                converted["content"] = ""
            #

            result.append(converted)
        #
        elif (role == "user"  and  isinstance(content, list)):

            user_text    = []
            tool_results = []

            for block in content:

                if (_compat_field(block, "type") == "tool_result"):

                    tool_results.append({
                        "role"         : "tool",
                        "tool_call_id" : _compat_field(block, "tool_use_id"),
                        "content"      : _compat_text(_compat_field(block, "content", "")),
                    })
                #
                else:

                    user_text.append(_compat_text(block))
                #
            #

            result.extend(tool_results)

            if (user_text):

                result.append({"role": "user", "content": "\n".join(user_text)})
            #
        #
        else:

            result.append({"role": role, "content": _compat_text(content)})
        #
    #

    return (result)
#
###################################################################################################
###################################################################################################
################################################################################################### OpenAI Message Adapter
#
class _NotebookMessages:

    """
        Expose messages.create on an OpenAI-compatible chat-completions client.

        Construct function-tool definitions, convert responses to the chapter's
        expected block objects, and translate token counts and stop reasons.
    """

    def __init__(self, client) -> None:

        """
            Keep the provider client used by subsequent model requests.
        """

        self._client = client
    #
    #################################################################  Public API
    #
    def create(
            self,
            *,
            model      : str,
            messages   : list,
            system     : str|list|None = None,
            tools      : list|None     = None,
            max_tokens : int|None      = None,
            **kwargs,
        ) -> object:

        """
            Send an OpenAI-compatible request and return Anthropic-shaped blocks.

            Tool requests remain associated with their call identifiers.
            The translation preserves the original lesson's model/tool loop.
        """

        request = {"model": model, "messages": _compat_history(messages, system)}

        if (tools):

            request["tools"] = [
                {
                    "type"     : "function",
                    "function" : {
                        "name"        : tool["name"],
                        "description" : tool.get("description", ""),
                        "parameters"  : tool.get(
                            "input_schema", {"type": "object", "properties": {}}
                        ),
                    },
                }
                for tool in tools
            ]
        #
        if (max_tokens is not None):

            request["max_tokens"] = max_tokens
        #

        response = self._client.chat.completions.create(**request)
        choice   = response.choices[0]
        reply    = choice.message
        blocks   = []

        if (reply.content):

            blocks.append(_compat_types.SimpleNamespace(
                type = "text",
                text = reply.content,
            ))
        #

        for call in (reply.tool_calls or []):

            try:

                arguments = _compat_json.loads(call.function.arguments or "{}")

                if (not isinstance(arguments, dict)):

                    arguments = {}
                #
            #
            except (ValueError, TypeError):

                arguments = {}
            #

            blocks.append(_compat_types.SimpleNamespace(
                type  = "tool_use",
                id    = call.id,
                name  = call.function.name,
                input = arguments,
            ))
        #

        usage       = getattr(response, "usage", None)
        stop_reason = (
            ("max_tokens") if (choice.finish_reason == "length")
            else (("tool_use") if (reply.tool_calls) else ("end_turn"))
        )

        return (_compat_types.SimpleNamespace(
            content     = blocks,
            stop_reason = stop_reason,
            usage       = _compat_types.SimpleNamespace(
                input_tokens  = getattr(usage, "prompt_tokens", 0)  or  0,
                output_tokens = getattr(usage, "completion_tokens", 0)  or  0,
            ),
        ))
    #
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Client
#
class _NotebookAnthropic:

    """
        Provide a legacy chapter client backed by the OpenCode endpoint.

        The name is retained for import compatibility; requests are always sent
        through the OpenAI Python SDK and use the notebook's configured API key.
    """

    def __init__(self, base_url:str|None=None, **kwargs) -> None:

        """
            Initialize the OpenAI client and expose the message adapter.
        """

        url = (_nb_os.getenv("OPENCODE_BASE_URL")  or  base_url  or  _notebook_url)

        self._client = OpenAI(
            api_key         = _notebook_key,
            base_url        = url,
            default_headers = {
                "x-opencode-session" : f"ses_{uuid.uuid4().hex}",
                "User-Agent"        : "learn-claude-code/1.0",
            },
        )
        self.messages = _NotebookMessages(self._client)
    #
#


_anthropic_compat           = _compat_types.ModuleType("anthropic")
_anthropic_compat.Anthropic = _NotebookAnthropic
_compat_sys.modules["anthropic"] = _anthropic_compat
#


In [ ]:
#!/usr/bin/env python3
"""
s10_task_system.py - Task System

    .tasks/
      task_a1b2c3d4.json  {status: completed, blockedBy: []}
      task_e5f6a7b8.json  {status: pending, blockedBy: [task_a1b2c3d4]}
      task_11223344.json  {status: pending, blockedBy: [task_e5f6a7b8]}

    Dependency graph:

    +-----------+      +-----------+      +-----------+
    | schema    | ---> | API       | ---> | tests     |
    | completed |      | pending   |      | pending   |
    +-----------+      +-----------+      +-----------+

    can_start(API) is true because schema is completed.

    Task lifecycle:

    pending --claim_task--> in_progress --complete_task--> completed
"""


In [ ]:

#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import glob
import json
import os
import re
import secrets
import subprocess
from dataclasses import asdict, dataclass
from pathlib import Path

try:

    import readline

    readline.parse_and_bind("set bind-tty-special-chars off")
    readline.parse_and_bind("set input-meta on")
    readline.parse_and_bind("set output-meta on")
    readline.parse_and_bind("set convert-meta off")
#
except ImportError:

    pass
#

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
if (os.getenv("ANTHROPIC_BASE_URL")):

    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)
#

WORKDIR = Path.cwd()
client  = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL   = os.environ["MODEL_ID"]

ENVIRONMENT_PROMPT = (
    ("Windows: the bash tool runs through cmd.exe; use cmd.exe syntax, not Unix "
    "Bash or PowerShell syntax, and prefer dedicated file tools for file operations")
    if (os.name == "nt")
    else ("Unix-like: the bash tool runs the system shell")
)
SYSTEM             = (
    f"You are a coding agent at {WORKDIR}. Environment: {ENVIRONMENT_PROMPT}. "
    "Use task tools to track dependencies and progress. Create all task nodes "
    "first. After create_task returns runtime-generated IDs, use update_task "
    "with those exact IDs to add dependencies."
)


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### New in s10: persistent task records
#

TASKS_DIR       = WORKDIR / ".tasks"
TASK_ID_PATTERN = re.compile(r"^task_[0-9a-f]{8}$")


@dataclass
class Task:

    id          : str
    subject     : str
    description : str
    status      : str
    owner       : str|None
    blockedBy   : list[str]
#


class TaskStore:

    #
    #################################################################  Inits
    #
    def __init__(self, directory:Path):

        self.directory = directory
    #

    #
    #################################################################  Internals
    #
    def _root(self, create:bool=False) -> Path:

        if (create):

            self.directory.mkdir(parents=True, exist_ok=True)
        #
        root = self.directory.resolve()
        if (not root.is_relative_to(WORKDIR.resolve())):

            raise (ValueError("Task store escapes the workspace"))
        #
        return (root)
    #

    #
    #################################################################  Internals
    #
    def _path(self, task_id:str, create_root:bool=False) -> Path:

        if (not isinstance(task_id, str)  or  not TASK_ID_PATTERN.fullmatch(task_id)):

            raise (ValueError(f"Invalid task ID: {task_id!r}"))
        #
        root = self._root(create=create_root)
        path = (root / f"{task_id}.json").resolve()
        if (not path.is_relative_to(root)):

            raise (ValueError(f"Invalid task ID: {task_id!r}"))
        #
        return (path)
    #

    #
    #################################################################  Public API
    #
    def exists(self, task_id:str) -> bool:

        return (self._path(task_id).is_file())
    #

    #
    #################################################################  Public API
    #
    def create(self, subject:str, description:str="") -> Task:

        subject = subject.strip()
        if (not subject):

            raise (ValueError("Task subject cannot be empty"))
        #

        self._root(create=True)
        for _ in range(100):

            task = Task(
                id          = f"task_{secrets.token_hex(4)}",
                subject     = subject,
                description = description,
                status      = "pending",
                owner       = None,
                blockedBy   = [],
            )
            try:

                with self._path(task.id, create_root=True).open(
                    "x", encoding="utf-8"
                ) as handle:

                    json.dump(asdict(task), handle, indent=2)
                #
                return (task)
            #
            except FileExistsError:

                continue
            #
        #
        raise (RuntimeError("Could not allocate a unique task ID"))
    #

    #
    #################################################################  Internals
    #
    def _depends_on(self, task_id:str, target_id:str) -> bool:

        """Return whether task_id transitively depends on target_id."""
        pending = [task_id]
        visited = set()
        while (pending):

            current = pending.pop()
            if (current == target_id):

                return (True)
            #
            if (current in visited):

                continue
            #
            visited.add(current)
            pending.extend(self.load(current).blockedBy)
        #
        return (False)
    #

    #
    #################################################################  Public API
    #
    def update_dependencies(self, task_id:str, add_blocked_by:list[str]) -> Task:

        if (not isinstance(add_blocked_by, list)):

            raise (ValueError("addBlockedBy must be a list of task IDs"))
        #

        task = self.load(task_id)
        if (task.status != "pending"  or  task.owner is not None):

            raise (ValueError(
                f"Task {task_id} dependencies can only be updated while "
                "pending and unowned"
            ))
        #

        dependencies = list(dict.fromkeys(add_blocked_by))
        for dependency in dependencies:

            if (dependency == task_id):

                raise (ValueError("Task cannot depend on itself"))
            #
            if (not self.exists(dependency)):

                raise (ValueError(f"Dependency not found: {dependency}"))
            #
            if (dependency not in task.blockedBy  and  self._depends_on(
                dependency, task_id
            )):

                raise (ValueError(
                    f"Dependency cycle detected: {task_id} -> {dependency}"
                ))
            #
        #

        task.blockedBy.extend(
            dependency for dependency in dependencies
            if dependency not in task.blockedBy
        )
        self.save(task)
        return (task)
    #

    def save(self, task:Task) -> None:

        self._path(task.id, create_root=True).write_text(
            json.dumps(asdict(task), indent=2),
            encoding="utf-8",
        )
    #

    def load(self, task_id:str) -> Task:

        data = json.loads(self._path(task_id).read_text(encoding="utf-8"))
        task = Task(**data)
        if (task.id != task_id):

            raise (ValueError(f"Task file ID does not match {task_id}"))
        #
        if (task.status not in ("pending", "in_progress", "completed")):

            raise (ValueError(f"Invalid task status: {task.status}"))
        #
        return (task)
    #

    def list(self) -> list[Task]:

        if (not self.directory.exists()):

            return ([])
        #
        root = self._root()
        return ([self.load(path.stem)
                for path in sorted(root.glob("task_*.json"))])
    #
#


TASKS = TaskStore(TASKS_DIR)




In [ ]:
def create_task(subject:str, description:str="") -> Task:

    return (TASKS.create(subject, description))
#


def update_task(task_id:str, addBlockedBy:list[str]) -> Task:

    return (TASKS.update_dependencies(task_id, addBlockedBy))
#


def load_task(task_id:str) -> Task:

    return (TASKS.load(task_id))
#


def list_tasks() -> list[Task]:

    return (TASKS.list())
#


def get_task(task_id:str) -> str:

    return (json.dumps(asdict(load_task(task_id)), indent=2))
#


def incomplete_dependencies(task:Task) -> list[str]:

    incomplete = []
    for dependency in task.blockedBy:

        try:

            if (load_task(dependency).status != "completed"):

                incomplete.append(dependency)
            #
        #
        except (FileNotFoundError, ValueError):

            incomplete.append(dependency)
        #
    #
    return (incomplete)
#


def can_start(task_id:str) -> bool:

    return (not incomplete_dependencies(load_task(task_id)))
#


def claim_task(task_id:str, owner:str="agent") -> str:

    task = load_task(task_id)
    if (task.status != "pending"):

        return (f"Task {task_id} is {task.status}, cannot claim")
    #
    dependencies = incomplete_dependencies(task)
    if (dependencies):

        return (f"Blocked by: {dependencies}")
    #
    task.owner  = owner
    task.status = "in_progress"
    TASKS.save(task)
    print(f"  [claim] {task.subject} -> in_progress (owner: {owner})")
    return (f"Claimed {task.id} ({task.subject})")
#


def complete_task(task_id:str, owner:str="agent") -> str:

    task = load_task(task_id)
    if (task.status != "in_progress"):

        return (f"Task {task_id} is {task.status}, cannot complete")
    #
    if (task.owner != owner):

        return (f"Task {task_id} is owned by {task.owner}, not {owner}")
    #
    ready_before = {
        candidate.id
        for candidate in list_tasks()
        if candidate.status == "pending"
        and  candidate.blockedBy
        and  can_start(candidate.id)
    }
    task.status  = "completed"
    TASKS.save(task)
    unblocked = [candidate.subject for candidate in list_tasks()
                 if candidate.status == "pending"
                 and  candidate.blockedBy
                 and  candidate.id not in ready_before
                 and  can_start(candidate.id)]
    print(f"  [complete] {task.subject}")
    message = f"Completed {task.id} ({task.subject})"
    if (unblocked):

        message += f"\nUnblocked: {', '.join(unblocked)}"
        print(f"  [unblocked] {', '.join(unblocked)}")
    #
    return (message)


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### From s04: tool implementations
#

def run_bash(command:str) -> str:

    try:

        result = subprocess.run(
            command,
            shell          = True,
            cwd            = WORKDIR,
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 120,
        )
        output = (result.stdout + result.stderr).strip()
        return ((output[:50000]) if (output) else ("(no output)"))
    #
    except subprocess.TimeoutExpired:

        return ("Error: Timeout (120s)")
    #
#


def run_read(path:str, limit:int|None=None) -> str:

    try:

        lines = (WORKDIR / path).resolve().read_text(encoding="utf-8").splitlines()
        if (limit  and  limit < len(lines)):

            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        #
        return ("\n".join(lines))
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


def run_write(path:str, content:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding="utf-8")
        return (f"Wrote {len(content)} bytes to {path}")
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


def run_edit(path:str, old_text:str, new_text:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        text      = file_path.read_text(encoding="utf-8")
        if (old_text not in text):

            return (f"Error: text not found in {path}")
        #
        file_path.write_text(text.replace(old_text, new_text, 1), encoding="utf-8")
        return (f"Edited {path}")
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


def run_glob(pattern:str) -> str:

    try:

        matches = sorted({
            match
            for match in glob.glob(pattern, root_dir=WORKDIR, recursive=True)
            if (WORKDIR / match).resolve().is_relative_to(WORKDIR)
        })
        shown   = matches[:200]
        if (len(matches) > 200):

            shown.append("... (more matches omitted; narrow the pattern)")
        #
        return (("\n".join(shown)) if (shown) else ("(no matches)"))
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


def run_create_task(subject:str, description:str="") -> str:

    task = create_task(subject, description)
    print(f"  [create] {task.subject}")
    return (f"Created {task.id}: {task.subject}")
#


def run_update_task(task_id:str, addBlockedBy:list[str]) -> str:

    task         = update_task(task_id, addBlockedBy)
    dependencies = ", ".join(task.blockedBy)  or  "(none)"
    print(f"  [update] {task.subject} blockedBy: {dependencies}")
    return (f"Updated {task.id} blockedBy: {dependencies}")
#


def run_list_tasks() -> str:

    tasks = list_tasks()
    if (not tasks):

        return ("No tasks. Use create_task to add some.")
    #
    lines = []
    for task in tasks:

        marker       = {
            "pending"     : "[ ]",
            "in_progress" : "[>]",
            "completed"   : "[x]",
        }.get(task.status, "[?]")
        dependencies = (
            (f" (blockedBy: {', '.join(task.blockedBy)})")
            if (task.blockedBy) else ("")
        )
        owner        = (f" [{task.owner}]") if (task.owner) else ("")
        lines.append(
            f"{marker} {task.id}: {task.subject} "
            f"[{task.status}]{owner}{dependencies}"
        )
    #
    return ("\n".join(lines))
#


def run_get_task(task_id:str) -> str:

    return (get_task(task_id))
#


def run_claim_task(task_id:str) -> str:

    return (claim_task(task_id, owner="agent"))
#


def run_complete_task(task_id:str) -> str:

    return (complete_task(task_id, owner="agent"))
#


TOOLS = [
    {
        "name"         : "bash",
        "description"  : "Run a shell command.",
        "input_schema" : {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
    },
    {
        "name"         : "read_file",
        "description"  : "Read file contents.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "limit": {"type": "integer"}}, "required": ["path"]},
    },
    {
        "name"         : "write_file",
        "description"  : "Write content to a file.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "content": {"type": "string"}}, "required": ["path", "content"]},
    },
    {
        "name"         : "edit_file",
        "description"  : "Replace exact text in a file once.",
        "input_schema" : {"type": "object", "properties": {"path": {"type": "string"}, "old_text": {"type": "string"}, "new_text": {"type": "string"}}, "required": ["path", "old_text", "new_text"]},
    },
    {
        "name"         : "glob",
        "description"  : "Find files matching a glob pattern; ** matches recursively.",
        "input_schema" : {"type": "object", "properties": {"pattern": {"type": "string"}}, "required": ["pattern"]},
    },
    {
        "name"         : "create_task",
        "description"  : "Create a task and return its runtime-generated ID.",
        "input_schema" : {"type": "object", "properties": {"subject": {"type": "string"}, "description": {"type": "string"}}, "required": ["subject"], "additionalProperties": False},
    },
    {
        "name"         : "update_task",
        "description"  : "Add dependencies using IDs returned by create_task.",
        "input_schema" : {"type": "object", "properties": {"task_id": {"type": "string", "pattern": "^task_[0-9a-f]{8}$"}, "addBlockedBy": {"type": "array", "items": {"type": "string", "pattern": "^task_[0-9a-f]{8}$"}, "minItems": 1}}, "required": ["task_id", "addBlockedBy"], "additionalProperties": False},
    },
    {
        "name"         : "list_tasks",
        "description"  : "List tasks with status, owner, and dependencies.",
        "input_schema" : {"type": "object", "properties": {}},
    },
    {
        "name"         : "get_task",
        "description"  : "Get a task by ID.",
        "input_schema" : {"type": "object", "properties": {"task_id": {"type": "string"}}, "required": ["task_id"]},
    },
    {
        "name"         : "claim_task",
        "description"  : "Claim a pending task whose dependencies are complete.",
        "input_schema" : {"type": "object", "properties": {"task_id": {"type": "string"}}, "required": ["task_id"]},
    },
    {
        "name"         : "complete_task",
        "description"  : "Complete the task claimed by this agent.",
        "input_schema" : {"type": "object", "properties": {"task_id": {"type": "string"}}, "required": ["task_id"]},
    },
]

TOOL_HANDLERS = {
    "bash"          : run_bash,
    "read_file"     : run_read,
    "write_file"    : run_write,
    "edit_file"     : run_edit,
    "glob"          : run_glob,
    "create_task"   : run_create_task,
    "update_task"   : run_update_task,
    "list_tasks"    : run_list_tasks,
    "get_task"      : run_get_task,
    "claim_task"    : run_claim_task,
    "complete_task" : run_complete_task,
}


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### From s04: hooks and permission checks
#

HOOKS = {"UserPromptSubmit": [], "PreToolUse": [], "PostToolUse": [], "Stop": []}


def register_hook(event:str, callback):

    HOOKS[event].append(callback)
#


def trigger_hooks(event:str, *args):

    for callback in HOOKS[event]:

        result = callback(*args)
        if (result is not None):

            return (result)
        #
    #
    return (None)
#


DENY_LIST                = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "dd if="]
DESTRUCTIVE_COMMAND_WORD = re.compile(
    r"(?i)(?:^|[;&|()\n])\s*(?:rm|del)(?=\s|$|[;&|()])"
)
DESTRUCTIVE              = ["rm ", "> /etc/", "chmod 777"]


def contains_destructive_command(command:str) -> bool:

    return (bool(DESTRUCTIVE_COMMAND_WORD.search(command)))
#


def permission_hook(block):

    if (block.name == "bash"):

        command = block.input.get("command", "")
        for pattern in DENY_LIST:

            if (pattern in command):

                print(f"\n\033[31m[blocked] '{pattern}'\033[0m")
                return ("Permission denied by deny list")
            #
        #
        if (contains_destructive_command(command)  or  any(
            keyword in command for keyword in DESTRUCTIVE
        )):

            print("\n\033[33m[permission] Potentially destructive command\033[0m")
            print(f"   Tool: {block.name}({block.input})")
            choice = input("   Allow? [y/N] ").strip().lower()
            if (choice not in ("y", "yes")):

                return ("Permission denied by user")
            #
        #
    #

    if (block.name in ("read_file", "write_file", "edit_file")):

        path = block.input.get("path", "")
        if (not (WORKDIR / path).resolve().is_relative_to(WORKDIR)):

            print("\n\033[33m[permission] Access outside workspace\033[0m")
            print(f"   Tool: {block.name}({block.input})")
            choice = input("   Allow? [y/N] ").strip().lower()
            if (choice not in ("y", "yes")):

                return ("Permission denied by user")
            #
        #
    #
    return (None)
#


def log_hook(block):

    preview = str(list(block.input.values())[:2])[:60]
    print(f"\033[90m[HOOK] {block.name}({preview})\033[0m")
    return (None)
#


def large_output_hook(block, output):

    if (len(str(output)) > 100000):

        print(
            f"\033[33m[HOOK] Large output from {block.name}: "
            f"{len(str(output))} chars\033[0m"
        )
    #
    return (None)
#


def context_hook(query:str):

    print(f"\033[90m[HOOK] UserPromptSubmit: working in {WORKDIR}\033[0m")
    return (None)
#


def summary_hook(messages:list):

    tool_count = sum(
        1
        for message in messages
        for block in (
            (message.get("content"))
            if (isinstance(message.get("content"), list))
            else ([])
        )
        if isinstance(block, dict)  and  block.get("type") == "tool_result"
    )
    print(f"\033[90m[HOOK] Stop: session used {tool_count} tool calls\033[0m")
    return (None)
#


register_hook("UserPromptSubmit", context_hook)
register_hook("PreToolUse", permission_hook)
register_hook("PreToolUse", log_hook)
register_hook("PostToolUse", large_output_hook)
register_hook("Stop", summary_hook)


def execute_tool(block) -> str:

    blocked = trigger_hooks("PreToolUse", block)
    if (blocked):

        return (str(blocked))
    #

    handler = TOOL_HANDLERS.get(block.name)
    try:

        output = (handler(**block.input)) if (handler) else (f"Unknown: {block.name}")
    #
    except Exception as error:

        output = f"Error: {error}"
    #

    trigger_hooks("PostToolUse", block, output)
    return (str(output))


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Agent loop
#

def agent_loop(messages:list):

    while (True):

        response = client.messages.create(
            model      = MODEL,
            system     = SYSTEM,
            messages   = messages,
            tools      = TOOLS,
            max_tokens = 8000,
        )
        messages.append({"role": "assistant", "content": response.content})

        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if (not tool_calls):

            force = trigger_hooks("Stop", messages)
            if (force):

                messages.append({"role": "user", "content": force})
                continue
            #
            return
        #

        results = []
        for block in tool_calls:

            output = execute_tool(block)
            results.append({
                "type"        : "tool_result",
                "tool_use_id" : block.id,
                "content"     : output,
            })
        #
        messages.append({"role": "user", "content": results})
    #
#


if (RUN_AGENT_INTERACTIVELY):

    print("s10: Task System - dependencies and task state")
    print("Enter a question, press Enter to send. Type q to quit.\n")

    history = []
    while (True):

        try:

            # \001/\002 tell Readline the ANSI escapes have zero display width.
            query = input("\001\033[36m\002s10 >> \001\033[0m\002")
        #
        except (EOFError, KeyboardInterrupt):

            break
        #
        if (query.strip().lower() in ("q", "exit", "")):

            break
        #
        trigger_hooks("UserPromptSubmit", query)
        history.append({"role": "user", "content": query})
        agent_loop(history)
        for block in history[-1]["content"]:

            if (getattr(block, "type", None) == "text"):

                print(block.text)
            #
        #
        print()
    #
#
